# 期現套利回測報告：可重跑分析 Notebook

## tl;dr

這個 Notebook 只讀取一月至七月同一次連續留倉回測所保存的 HBT CSV，不重跑回測。執行後會顯示排除異常日期後的 ROI、Sharpe proxy、損益組成、資金占用與資料品質檢查，並輸出專業報告使用的稽核表。

## 分析背景與方法

### 重要假設

- 舊資金周轉 ROI 仍保留作比較；另以 5,000 萬共用自有資金、期貨 20% 保證金與現股 40% 自備款，對已保存的候選成交做跨日資金約束重播。
- 依比例分配後，期貨資金上限約 1,666.7 萬、現股自備款上限約 3,333.3 萬，配對名目上限約 8,333.3 萬。
- HBT 會把前一日未平倉部位灌入下一交易日，持倉不在當日篩選名單時仍保留；舊期貨不換月，到期日殘倉視為錯誤。
- 資金占用同步跨日延續，但資金篩選仍是對已保存候選成交的事後重播，不是 HBT 下單前的全域資金風控。
- Sharpe proxy 使用 active-day ROI、252 日年化、無風險利率 0；不是連續投資組合權益曲線的標準 Sharpe。
- 排除日期與 `replot_filtered_results.ipynb` 一致。

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd
from IPython.display import Markdown, display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / 'future_spot').exists():
    WORKSPACE_ROOT = CURRENT_DIR
elif CURRENT_DIR.name == 'notebooks' and CURRENT_DIR.parent.name == 'future_spot':
    WORKSPACE_ROOT = CURRENT_DIR.parents[1]
else:
    raise FileNotFoundError('請從 repository root 或 future_spot/notebooks 執行')
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

import future_spot.arbitrage.backtest_report as backtest_report
importlib.reload(backtest_report)
from future_spot.arbitrage.backtest_report import build_backtest_report
from future_spot.arbitrage.result_replot import PlotInterval

## 資料

### 1. 設定既有輸出與排除日期

In [ ]:
OUTPUT_DIRS = [
    # 跨月留倉必須使用同一次連續 runner 的輸出，不能再拼接七個月度獨立回測。
    WORKSPACE_ROOT / 'future_spot/output/hbt_daily_full_market_20260101_20260731_latency_10ms',
]

REPORT_INTERVALS = [
    PlotInterval('2026-01', '2026-01-01', '2026-01-31'),
    PlotInterval('2026-02', '2026-02-01', '2026-02-28'),
    PlotInterval('2026-03', '2026-03-01', '2026-03-31'),
    PlotInterval('2026-04', '2026-04-01', '2026-04-30', ('2026-04-23',)),
    PlotInterval('2026-05', '2026-05-01', '2026-05-31', ('2026-05-04', '2026-05-19')),
    PlotInterval('2026-06', '2026-06-01', '2026-06-30', ('2026-06-24',)),
    PlotInterval('2026-07', '2026-07-01', '2026-07-31', ('2026-07-01', '2026-07-22', '2026-07-27')),
]

REPORT_OUTPUT_DIR = WORKSPACE_ROOT / 'future_spot/output/backtest_report_202601_202607'

TOTAL_CAPITAL = 50_000_000
FUTURES_MARGIN_RATE = 0.20
SPOT_EQUITY_RATE = 0.40
LEVERAGE = True  # False 時期貨與現股皆按 100% 自有資金
CARRY_POSITIONS = True  # 與 full-market runner 的跨日留倉模式一致

### 2. 建立報告資料與稽核證據

In [ ]:
report = build_backtest_report(
    OUTPUT_DIRS,
    REPORT_INTERVALS,
    REPORT_OUTPUT_DIR,
    total_capital=TOTAL_CAPITAL,
    futures_margin_rate=FUTURES_MARGIN_RATE,
    spot_equity_rate=SPOT_EQUITY_RATE,
    leverage=LEVERAGE,
    carry_positions=CARRY_POSITIONS,
)
h = report.headline
display(Markdown(
    f"**過濾後結果：** 含未平倉損益的總 PnL 為 NT${h['total_pnl_including_open']/1_000_000:.2f}M，"
    f"資金周轉 ROI 為 {h['total_roi']:.3%}，Sharpe proxy 為 {h['sharpe_proxy']:.2f}。"
    f"未平倉鎖定 PnL 占 {h['open_pnl_share']:.1%}。5,000 萬資金篩選接受 "
    f"{h['capital_entry_acceptance_rate']:.1%} 的候選進場；此結果為跨日持倉重播估計。"
))
headline_labels = {
    'period_start': '資料起日', 'period_end': '資料迄日',
    'observed_trade_days': '觀測交易日', 'active_roi_days': '有效 ROI 日',
    'entry_cash_out': '累計進場現金', 'realized_pnl': '已實現 PnL',
    'open_locked_pnl': '未平倉鎖定 PnL', 'total_pnl_including_open': '總 PnL（含未平倉）',
    'total_roi': '資金周轉 ROI', 'realized_roi': '已實現 ROI',
    'open_pnl_share': '未平倉 PnL 占比', 'sharpe_proxy': 'Sharpe proxy',
    'sortino_proxy': 'Sortino proxy', 'active_day_win_rate': 'active-day 勝率',
    'max_drawdown_twd': '最大回撤（NT$）', 'entries': '進場次數',
    'exits': '出場次數', 'open_pair_runs': '未平倉 pair-run',
    'completion_rate': '完成率', 'peak_daily_entry_cash': '單日進場現金峰值',
    'peak_daily_stuck_cash': '單日卡住現金峰值', 'summary_realized_pnl': 'Summary 已實現 PnL',
    'excluded_dates': '排除日期數', 'excluded_summary_pnl': '排除日期的 Summary PnL',
    'config_error_days': '設定錯誤日數', 'top_two_month_pnl_share': '前兩個月 PnL 占比',
    'capital_total': '共用自有資金', 'futures_margin_rate': '期貨保證金率',
    'spot_equity_rate': '現股自備款率', 'futures_capital_limit': '期貨資金上限',
    'spot_capital_limit': '現股自備款上限', 'matched_notional_limit': '配對名目上限',
    'capital_candidate_entries': '資金篩選候選進場', 'capital_accepted_entries': '資金篩選接受進場',
    'capital_rejected_entries': '資金不足拒絕進場', 'capital_entry_acceptance_rate': '候選進場接受率',
    'capital_filtered_realized_pnl': '資金篩選已實現 PnL',
    'capital_filtered_realized_roi': '資金篩選已實現 ROI', 'capital_peak_used': '資金使用峰值',
    'capital_peak_utilization': '資金使用率峰值', 'capital_replay_scope': '資金重播範圍',
}
headline_table = report.frame('headline_metrics').T.rename(columns={0: '數值'}).rename(index=headline_labels)
display(headline_table)

## 分析結果

### 3. 月度績效與資金占用

In [ ]:
monthly_columns = [
    'month', 'observed_trade_days', 'active_days', 'entry_cash_out', 'realized_pnl',
    'open_locked_pnl', 'total_pnl_including_open', 'total_roi', 'sharpe_proxy',
    'completion_rate',
]
monthly_labels = {
    'month': '月份', 'observed_trade_days': '觀測交易日', 'active_days': '有效 ROI 日',
    'entry_cash_out': '進場現金', 'realized_pnl': '已實現 PnL',
    'open_locked_pnl': '未平倉鎖定 PnL', 'total_pnl_including_open': '總 PnL',
    'total_roi': '資金周轉 ROI', 'sharpe_proxy': 'Sharpe proxy', 'completion_rate': '完成率',
}
symbol_labels = {
    'spot_symbol': '現貨代號', 'pair_runs': 'pair-run 數', 'entry_cash_out': '進場現金',
    'realized_pnl': '已實現 PnL', 'open_locked_pnl': '未平倉鎖定 PnL',
    'total_pnl_including_open': '總 PnL', 'open_pairs': '未平倉 pair-run',
    'total_roi': '資金周轉 ROI', 'open_pnl_share': '未平倉 PnL 占比',
}
display(report.frame('monthly_performance')[monthly_columns].rename(columns=monthly_labels))
display(report.frame('symbol_performance').head(15).rename(columns=symbol_labels))

capital_labels = {
    'total_capital': '共用自有資金', 'futures_margin_rate': '期貨保證金率',
    'spot_equity_rate': '現股自備款率', 'futures_capital_limit': '期貨資金上限',
    'spot_capital_limit': '現股自備款上限', 'matched_notional_limit': '配對名目上限',
    'candidate_entries': '候選進場', 'accepted_entries': '接受進場',
    'rejected_entries': '資金不足拒絕', 'entry_acceptance_rate': '接受率',
    'accepted_exits': '接受出場', 'ending_open_lot_days': '日終未平倉 lot-day',
    'capital_filtered_realized_pnl': '資金篩選已實現 PnL',
    'capital_filtered_realized_roi': '資金篩選已實現 ROI',
    'peak_spot_capital': '現股資金峰值', 'peak_futures_capital': '期貨資金峰值',
    'peak_total_capital': '總資金峰值', 'peak_capital_utilization': '總資金使用率峰值',
    'replay_scope': '重播範圍',
}
display(report.frame('capital_constraint_summary').T.rename(columns={0: '數值'}).rename(index=capital_labels))
display(report.frame('daily_capital_constraint').sort_values('peak_capital_utilization', ascending=False).head(15))

### 4. 資料品質與排除日期稽核

In [ ]:
coverage_labels = {
    'interval': '月份', 'attempted_trade_days': '嘗試交易日', 'config_success_days': '設定成功日',
    'config_error_days': '設定錯誤日', 'config_error_dates': '錯誤日期',
    'summary_days_before_exclusion': '排除前 summary 日', 'excluded_days_found': '排除日',
    'included_summary_days': '納入 summary 日', 'active_roi_days': '有效 ROI 日',
}
excluded_labels = {
    'interval': '月份', 'trade_date': '日期', 'found_in_summary': 'summary 中可找到',
    'summary_realized_pnl_removed': '移除的 summary PnL', 'filled_pairs_removed': '移除的成交配對',
    'second_leg_failures_removed': '移除的第二腿失敗', 'reason': '原因',
}
display(report.frame('validation_checks'))
display(report.frame('coverage').rename(columns=coverage_labels))
display(report.frame('excluded_dates').rename(columns=excluded_labels))
print(f'報告稽核資料目錄：{report.output_dir}')
print(f'標準報告 artifact：{report.output_dir / "artifact.json"}')

## 結論與後續行動

- ROI 與 Sharpe 看起來很強，但主要限制不是日內虧損，而是大量未平倉 pair-run、資金卡住，以及非固定本金的報酬口徑。
- 對外分享時必須同時呈現已實現 PnL、未平倉鎖定 PnL、完成率、資料排除與缺失日期。
- 新增的 5,000 萬資金模型會按期貨 20%／現股 40% 拒絕超額候選單；舊 `stuck cash` 僅保留作歷史現金流比較。
- 資金模型目前是逐日候選重播；下一步仍需建立跨日連續持倉、融資利息、每日市值評價、追繳與強制平倉的完整權益曲線。